# PyGeoFetch — 06: Real-World Workflows

End-to-end workflows combining PyGeoFetch data acquisition with geospatial analysis.

---
### What you'll learn
- NDVI time series over an agricultural area
- Bi-temporal change detection (urban expansion)
- Multi-sensor fusion (Sentinel-2 + Landsat)
- Cloud-free mosaic generation
- SAR + optical data fusion (Sentinel-1 + Sentinel-2)
- Automated weekly monitoring pipeline

---
> **Prerequisites:** `pip install rasterio geopandas matplotlib numpy scipy`

In [ ]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pygeofetch.models.download_task import DownloadOptions
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from collections import defaultdict

client = PyGeoFetch(log_level="WARNING")
for d in ["output/ndvi", "output/change", "output/fusion", "output/mosaic", "output/sar"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Ready — all output directories created")

---
## Workflow 1: NDVI Time Series (Vegetation Monitoring)

In [ ]:
# Iowa corn belt — ideal for NDVI monitoring
# Bbox: small agricultural area near Ames, Iowa
IOWA_BBOX = "-93.8,41.9,-93.5,42.1"

# Search full year to capture the growing season
query_ndvi = SearchQuery(
    bbox=BoundingBox.from_string(IOWA_BBOX),
    start_date="2024-01-01",
    end_date="2024-12-31",
    cloud_cover_max=20,
    satellites=["Sentinel-2"],
    max_results=100,
    sort_by="datetime",
    sort_ascending=True,
)

results_ndvi = client.search(query_ndvi, providers=["aws_earth", "planetary_computer"])
print(f"Found {len(results_ndvi)} Sentinel-2 scenes over Iowa (full year 2024)")

# Group by month
monthly = defaultdict(list)
for r in results_ndvi:
    dt = str(r.properties.get("datetime", ""))[:7] if r.properties else ""
    if dt:
        monthly[dt].append(r)

print("\nAvailability by month:")
for month in sorted(monthly):
    scenes = monthly[month]
    avg_cloud = np.mean([s.cloud_cover for s in scenes if s.cloud_cover is not None])
    bar = "█" * len(scenes)
    print(f"  {month}  {bar:<20} {len(scenes):>3} scenes  avg_cloud={avg_cloud:.0f}%")

In [ ]:
# Pick the clearest scene per month for time series
monthly_best = {}
for month, scenes in monthly.items():
    best = min(scenes, key=lambda s: s.cloud_cover if s.cloud_cover is not None else 999)
    monthly_best[month] = best

print(f"Selected {len(monthly_best)} scenes (1 per month):")
print(f"{'Month':<10} {'ID':<40} {'Cloud%':>7}")
print("-" * 60)
for month in sorted(monthly_best):
    s = monthly_best[month]
    print(f"  {month:<10} {s.id[:38]:<40} {s.cloud_cover:>7.0f}%")

In [ ]:
# Download the NDVI bands (B04=Red, B08=NIR) for all monthly scenes
scenes_to_dl = list(monthly_best.values())

options_ndvi = DownloadOptions(
    parallel=3,
    retry_attempts=3,
    resume=True,
    on_failure="skip",
    bands=["B04", "B08"],   # Red + NIR = NDVI
)

print(f"Downloading B04+B08 for {len(scenes_to_dl)} monthly scenes...")
print(f"Estimated size: ~{len(scenes_to_dl) * 100:.0f} MB (100 MB/scene)")

dl_results = client.download(
    scenes_to_dl,
    destination=Path("output/ndvi/raw"),
    options=options_ndvi,
)

succeeded = [r for r in dl_results if r.success]
total_mb = sum(r.bytes_downloaded for r in succeeded if r.bytes_downloaded) / (1024*1024)
print(f"\nDownloaded {len(succeeded)}/{len(scenes_to_dl)} scenes — {total_mb:.1f} MB total")

In [ ]:
# Calculate NDVI for each downloaded pair and build the time series
# NDVI = (NIR - Red) / (NIR + Red)  — values -1 to +1, vegetation > 0.3

try:
    import rasterio
    import rasterio.mask

    ndvi_timeseries = []  # (date, mean_ndvi, std_ndvi)
    ndvi_dir = Path("output/ndvi/raw")

    # Match B04 and B08 files by scene ID
    b04_files = sorted(ndvi_dir.rglob("*B04*.tif"))
    b08_files = sorted(ndvi_dir.rglob("*B08*.tif"))

    # Pair by matching stem
    def get_scene_id(path):
        parts = path.stem.split("_")
        return "_".join(parts[:4]) if len(parts) >= 4 else path.stem

    b04_by_scene = {get_scene_id(f): f for f in b04_files}
    b08_by_scene = {get_scene_id(f): f for f in b08_files}
    common_scenes = set(b04_by_scene) & set(b08_by_scene)

    print(f"Computing NDVI for {len(common_scenes)} scenes...")

    for scene_id in sorted(common_scenes):
        b04_path = b04_by_scene[scene_id]
        b08_path = b08_by_scene[scene_id]

        with rasterio.open(b04_path) as red_src:
            red = red_src.read(1).astype(np.float32)
            nodata = red_src.nodata

        with rasterio.open(b08_path) as nir_src:
            nir = nir_src.read(1).astype(np.float32)

        # Mask nodata
        mask = (red > 0) & (nir > 0)
        if nodata:
            mask &= (red != nodata) & (nir != nodata)

        red_m, nir_m = red[mask], nir[mask]
        denom = nir_m + red_m
        valid = denom > 0
        ndvi = np.where(valid, (nir_m[valid] - red_m[valid]) / denom[valid], np.nan)

        if len(ndvi) > 0:
            # Extract date from filename
            import re
            date_match = re.search(r'(\d{8})', b04_path.stem)
            if date_match:
                d = date_match.group(1)
                date = datetime(int(d[:4]), int(d[4:6]), int(d[6:8]))
                mean_ndvi = float(np.nanmean(ndvi))
                std_ndvi  = float(np.nanstd(ndvi))
                ndvi_timeseries.append((date, mean_ndvi, std_ndvi))
                print(f"  {date.strftime('%Y-%m-%d')}  mean_ndvi={mean_ndvi:.3f}  std={std_ndvi:.3f}")

    ndvi_timeseries.sort(key=lambda x: x[0])
    print(f"\nNDVI time series: {len(ndvi_timeseries)} points")

except ImportError:
    print("rasterio not installed — generating synthetic NDVI for demonstration")
    # Realistic synthetic growing season NDVI curve
    months = pd.date_range("2024-01-01", "2024-12-01", freq="MS")
    ndvi_curve = [0.12, 0.14, 0.18, 0.28, 0.55, 0.72, 0.80, 0.75, 0.52, 0.30, 0.16, 0.12]
    ndvi_timeseries = [
        (m.to_pydatetime(), v + np.random.normal(0, 0.02), 0.05)
        for m, v in zip(months, ndvi_curve)
    ]
    print("Using synthetic NDVI values for plotting demonstration")

In [ ]:
# Plot the NDVI time series
if ndvi_timeseries:
    dates = [x[0] for x in ndvi_timeseries]
    means = [x[1] for x in ndvi_timeseries]
    stds  = [x[2] for x in ndvi_timeseries]

    fig, ax = plt.subplots(figsize=(14, 5))

    ax.fill_between(dates,
                    [m - s for m, s in zip(means, stds)],
                    [m + s for m, s in zip(means, stds)],
                    alpha=0.2, color='green', label='±1 std')
    ax.plot(dates, means, 'o-', color='#2d7a2d', linewidth=2,
            markersize=7, label='Mean NDVI')

    # Add reference bands
    ax.axhspan(0.6, 1.0, alpha=0.06, color='darkgreen', label='Dense vegetation')
    ax.axhspan(0.2, 0.6, alpha=0.06, color='yellowgreen', label='Sparse vegetation')
    ax.axhspan(-0.1, 0.2, alpha=0.06, color='tan', label='Bare soil / urban')
    ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')

    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('NDVI', fontsize=12)
    ax.set_title('NDVI Time Series — Iowa Agricultural Area (2024)\n(Sentinel-2, 10m resolution)', fontsize=13)
    ax.set_ylim(-0.1, 1.0)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.xticks(rotation=30, ha='right')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('output/ndvi/ndvi_timeseries.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ NDVI time series saved → output/ndvi/ndvi_timeseries.png")

    # Summary stats
    peak_idx = np.argmax(means)
    print(f"\nGrowing season summary:")
    print(f"  Peak NDVI: {max(means):.3f} on {dates[peak_idx].strftime('%Y-%m-%d')}")
    print(f"  Min NDVI:  {min(means):.3f} on {dates[np.argmin(means)].strftime('%Y-%m-%d')}")
    print(f"  Mean NDVI: {np.mean(means):.3f}")

---
## Workflow 2: Bi-Temporal Change Detection

In [ ]:
# Urban expansion detection — compare 2020 vs 2024
# Houston, TX — one of the fastest-growing US cities
HOUSTON_BBOX = "-95.6,29.6,-95.2,30.0"

def search_best_scene(bbox, start, end, provider="aws_earth"):
    """Find the single clearest scene in a date range."""
    q = SearchQuery(
        bbox=BoundingBox.from_string(bbox),
        start_date=start,
        end_date=end,
        cloud_cover_max=10,
        max_results=20,
        sort_by="cloud_cover",
        sort_ascending=True,
    )
    results = client.search(q, providers=[provider])
    return results[0] if results else None

# Find best scenes for both periods
print("Searching for before/after scenes over Houston...")
scene_2020 = search_best_scene(HOUSTON_BBOX, "2020-06-01", "2020-09-30")
scene_2024 = search_best_scene(HOUSTON_BBOX, "2024-06-01", "2024-09-30")

if scene_2020 and scene_2024:
    print(f"\nBEFORE: {scene_2020.id}")
    date_2020 = str(scene_2020.properties.get("datetime", ""))[:10] if scene_2020.properties else "?"
    print(f"  Date: {date_2020}  Cloud: {scene_2020.cloud_cover}%")

    print(f"\nAFTER:  {scene_2024.id}")
    date_2024 = str(scene_2024.properties.get("datetime", ""))[:10] if scene_2024.properties else "?"
    print(f"  Date: {date_2024}  Cloud: {scene_2024.cloud_cover}%")
else:
    print("No suitable scenes found — try relaxing cloud cover threshold")

In [ ]:
# Download RGB + NIR for both epochs
change_options = DownloadOptions(
    parallel=2,
    resume=True,
    on_failure="skip",
    bands=["B02", "B03", "B04", "B08"],  # RGB + NIR
)

scenes_change = [s for s in [scene_2020, scene_2024] if s]
if scenes_change:
    print(f"Downloading {len(scenes_change)} scenes (RGB + NIR)...")
    dl = client.download(scenes_change, Path("output/change/raw"), change_options)
    for r in dl:
        if r.success:
            mb = r.bytes_downloaded / (1024*1024) if r.bytes_downloaded else 0
            print(f"  ✓ {r.data_id[:40]} — {mb:.1f} MB")
        else:
            print(f"  ✗ {r.error}")

In [ ]:
# Compute NDBI change (Normalized Difference Built-up Index)
# NDBI = (SWIR - NIR) / (SWIR + NIR) — higher values = more urban/built-up
# We'll use NDVI change as proxy: decrease in NDVI = urbanization

try:
    import rasterio
    from rasterio.enums import Resampling

    change_dir = Path("output/change/raw")

    # Find files from each epoch
    def find_band(directory, scene_id_fragment, band_code):
        matches = list(directory.rglob(f"*{band_code}*.tif"))
        if scene_id_fragment:
            matches = [m for m in matches if scene_id_fragment in str(m)]
        return matches[0] if matches else None

    # Try to compute NDVI change
    if scene_2020 and scene_2024:
        frag_2020 = scene_2020.id.split('_')[1][:8] if '_' in scene_2020.id else scene_2020.id[:8]
        frag_2024 = scene_2024.id.split('_')[1][:8] if '_' in scene_2024.id else scene_2024.id[:8]

        b04_2020 = find_band(change_dir, frag_2020, "B04")
        b08_2020 = find_band(change_dir, frag_2020, "B08")
        b04_2024 = find_band(change_dir, frag_2024, "B04")
        b08_2024 = find_band(change_dir, frag_2024, "B08")

        def compute_ndvi(red_path, nir_path):
            with rasterio.open(red_path) as rs, rasterio.open(nir_path) as ns:
                red = rs.read(1).astype(np.float32)
                nir = ns.read(1).astype(np.float32)
                # Resize nir to match red if needed
                if red.shape != nir.shape:
                    from rasterio.warp import reproject
                    nir_resampled = np.empty_like(red)
                    reproject(nir, nir_resampled, src_transform=ns.transform,
                              src_crs=ns.crs, dst_transform=rs.transform,
                              dst_crs=rs.crs, resampling=Resampling.bilinear)
                    nir = nir_resampled
            with np.errstate(divide='ignore', invalid='ignore'):
                ndvi = np.where((nir + red) > 0, (nir - red) / (nir + red), np.nan)
            return ndvi, rs.profile

        if all([b04_2020, b08_2020, b04_2024, b08_2024]):
            ndvi_2020, profile = compute_ndvi(b04_2020, b08_2020)
            ndvi_2024, _ = compute_ndvi(b04_2024, b08_2024)

            # Align arrays to same shape
            min_rows = min(ndvi_2020.shape[0], ndvi_2024.shape[0])
            min_cols = min(ndvi_2020.shape[1], ndvi_2024.shape[1])
            ndvi_change = ndvi_2024[:min_rows, :min_cols] - ndvi_2020[:min_rows, :min_cols]

            # Thresholds for change detection
            vegetation_gain = ndvi_change > 0.2   # New vegetation
            vegetation_loss = ndvi_change < -0.2  # Lost vegetation (urbanization)

            total_pixels = np.sum(~np.isnan(ndvi_change))
            gain_pct  = 100 * np.sum(vegetation_gain) / total_pixels
            loss_pct  = 100 * np.sum(vegetation_loss) / total_pixels
            stable_pct = 100 - gain_pct - loss_pct

            print(f"Change detection results (2020 → 2024):")
            print(f"  Vegetation gain (green areas): {gain_pct:.1f}%")
            print(f"  Vegetation loss (urban/built): {loss_pct:.1f}%")
            print(f"  No change:                     {stable_pct:.1f}%")

            # Save change map
            profile.update(dtype='float32', count=1, nodata=np.nan)
            with rasterio.open('output/change/ndvi_change.tif', 'w', **profile) as dst:
                dst.write(ndvi_change.astype('float32'), 1)
            print("\n✓ Change raster saved → output/change/ndvi_change.tif")

            # Plot
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            kw = dict(vmin=-0.3, vmax=0.3, cmap='RdYlGn')

            im0 = axes[0].imshow(ndvi_2020[:min_rows, :min_cols], **kw)
            axes[0].set_title(f'NDVI 2020\n{date_2020}', fontsize=12)
            axes[0].axis('off')

            im1 = axes[1].imshow(ndvi_2024[:min_rows, :min_cols], **kw)
            axes[1].set_title(f'NDVI 2024\n{date_2024}', fontsize=12)
            axes[1].axis('off')

            im2 = axes[2].imshow(ndvi_change, vmin=-0.5, vmax=0.5,
                                  cmap='RdYlGn')
            axes[2].set_title('NDVI Change\n(green=gain, red=loss)', fontsize=12)
            axes[2].axis('off')
            fig.colorbar(im2, ax=axes[2], fraction=0.04)

            plt.suptitle('Bi-Temporal Change Detection — Houston, TX (2020→2024)', fontsize=14, y=1.01)
            plt.tight_layout()
            plt.savefig('output/change/change_detection.png', dpi=150, bbox_inches='tight')
            plt.show()
        else:
            print("Could not find all required files — make sure downloads completed")

except ImportError:
    print("rasterio not installed — showing synthetic change statistics")
    print("\nSimulated change detection results (Houston 2020→2024):")
    print("  Vegetation gain: 3.2%")
    print("  Vegetation loss: 8.7% (urban expansion)")
    print("  No significant change: 88.1%")
    print("\n  Install rasterio for real computation: pip install rasterio")

---
## Workflow 3: Multi-Sensor Fusion (Sentinel-2 + Landsat)

In [ ]:
# Combine Sentinel-2 (10m, 5-day revisit) and Landsat-9 (30m, 16-day revisit)
# to maximize temporal density while preserving spatial resolution

NYC_BBOX = "-74.1,40.6,-73.7,40.9"

# Search both sensors simultaneously
q_fusion = SearchQuery(
    bbox=BoundingBox.from_string(NYC_BBOX),
    start_date="2024-04-01",
    end_date="2024-09-30",
    cloud_cover_max=25,
    max_results=100,
    sort_by="datetime",
    sort_ascending=True,
)

results_fusion = client.search(q_fusion, providers=["aws_earth", "planetary_computer"])

# Split by satellite family
sentinel2_scenes = [r for r in results_fusion if r.satellite and "sentinel-2" in r.satellite.lower()]
landsat_scenes   = [r for r in results_fusion if r.satellite and "landsat" in r.satellite.lower()]
other_scenes     = [r for r in results_fusion if r not in sentinel2_scenes + landsat_scenes]

print(f"Multi-sensor results over NYC (Apr-Sep 2024):")
print(f"  Sentinel-2: {len(sentinel2_scenes)} scenes")
print(f"  Landsat:    {len(landsat_scenes)} scenes")
print(f"  Other:      {len(other_scenes)} scenes")
print(f"  Total:      {len(results_fusion)} scenes")

In [ ]:
# Build a combined acquisition timeline
all_dates = []
for r in results_fusion:
    dt_str = str(r.properties.get("datetime", ""))[:10] if r.properties else ""
    if dt_str:
        try:
            dt = datetime.strptime(dt_str, "%Y-%m-%d")
            family = ("Sentinel-2" if r.satellite and "sentinel" in r.satellite.lower()
                      else "Landsat" if r.satellite and "landsat" in r.satellite.lower()
                      else "Other")
            all_dates.append((dt, family, r.cloud_cover or 0))
        except ValueError:
            pass

all_dates.sort(key=lambda x: x[0])

# Plot timeline
fig, ax = plt.subplots(figsize=(16, 4))

colors = {"Sentinel-2": "#1a6eb5", "Landsat": "#e65c00", "Other": "#888"}
y_pos  = {"Sentinel-2": 1.0, "Landsat": 0.0, "Other": 0.5}

for dt, family, cloud in all_dates:
    y = y_pos[family]
    alpha = max(0.3, 1 - cloud / 100)
    ax.scatter(dt, y, c=colors[family], s=80, alpha=alpha,
               edgecolors='none', zorder=3)

# Compute gap analysis
unique_days = sorted(set(x[0].date() for x in all_dates))
if len(unique_days) > 1:
    gaps = [(unique_days[i+1] - unique_days[i]).days for i in range(len(unique_days)-1)]
    max_gap = max(gaps)
    avg_gap = sum(gaps) / len(gaps)
else:
    max_gap = avg_gap = 0

ax.set_yticks([0, 1])
ax.set_yticklabels(["Landsat (30m)", "Sentinel-2 (10m)"], fontsize=11)
ax.set_xlabel("Date", fontsize=11)
ax.set_title(f"Multi-Sensor Acquisition Timeline — NYC (Apr–Sep 2024)\n"
             f"{len(results_fusion)} total scenes · Avg gap: {avg_gap:.1f} days · Max gap: {max_gap} days",
             fontsize=12)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=30, ha='right')

for name, color in colors.items():
    if name != "Other":
        ax.scatter([], [], c=color, s=80, label=name)
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
ax.set_ylim(-0.5, 1.5)

plt.tight_layout()
plt.savefig('output/fusion/acquisition_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Timeline saved → output/fusion/acquisition_timeline.png")
print(f"\nWith multi-sensor fusion: avg revisit = {avg_gap:.1f} days")
print(f"  Sentinel-2 only:  ~5 days")
print(f"  Landsat only:     ~16 days")
print(f"  Combined:         ~{avg_gap:.1f} days ({'better' if avg_gap < 5 else 'similar'} than single-sensor)")

---
## Workflow 4: Cloud-Free Mosaic

In [ ]:
# Strategy: collect all scenes from a month, pick clear pixels from each
# to fill cloud gaps — produces a cloud-free monthly composite

q_mosaic = SearchQuery(
    bbox=BoundingBox.from_string(NYC_BBOX),
    start_date="2024-06-01",
    end_date="2024-06-30",
    cloud_cover_max=40,   # Accept cloudier scenes — we'll pick clear pixels
    max_results=30,
    sort_by="cloud_cover",
    sort_ascending=True,
)

results_mosaic = client.search(q_mosaic, providers=["aws_earth"])
print(f"June 2024 scenes: {len(results_mosaic)}")

cloud_vals = [r.cloud_cover for r in results_mosaic if r.cloud_cover is not None]
print(f"Cloud cover range: {min(cloud_vals):.0f}% – {max(cloud_vals):.0f}%")
print(f"Mean cloud cover: {np.mean(cloud_vals):.0f}%")
print()
print("Mosaic strategy:")
print("  1. Download SCL (Scene Classification Layer) for each scene")
print("  2. Stack all scenes, use SCL to identify clear pixels")
print("  3. For each pixel: take median of all cloud-free observations")
print("  4. Fill remaining gaps with nearest-neighbour from cleanest scene")

In [ ]:
# Simulate mosaic quality analysis (real computation requires rasterio)
n_scenes = len(results_mosaic)
avg_cloud = np.mean(cloud_vals) / 100  # as fraction

# P(pixel is cloud-free in at least 1 of N scenes) = 1 - avg_cloud^N
coverage = []
for n in range(1, n_scenes + 1):
    p_cloud_free = 1 - (avg_cloud ** n)
    coverage.append(p_cloud_free * 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, n_scenes + 1), coverage, 'b-o', markersize=5)
ax.axhline(95, color='green', linestyle='--', label='95% coverage threshold')
ax.axhline(99, color='red', linestyle='--', label='99% coverage threshold')

n_for_95 = next((i+1 for i, c in enumerate(coverage) if c >= 95), n_scenes)
n_for_99 = next((i+1 for i, c in enumerate(coverage) if c >= 99), n_scenes)

ax.scatter([n_for_95], [coverage[n_for_95-1]], c='green', s=100, zorder=5)
ax.scatter([n_for_99], [coverage[n_for_99-1]], c='red', s=100, zorder=5)

ax.set_xlabel('Number of scenes in mosaic', fontsize=12)
ax.set_ylabel('% pixels with at least one clear observation', fontsize=12)
ax.set_title(f'Cloud-Free Mosaic Coverage Analysis\n'
             f'(avg cloud cover: {avg_cloud*100:.0f}%, {n_scenes} available scenes)',
             fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 101)

plt.tight_layout()
plt.savefig('output/mosaic/coverage_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Coverage analysis saved")
print(f"\nMinimum scenes needed:")
print(f"  For 95% coverage: {n_for_95} scenes")
print(f"  For 99% coverage: {n_for_99} scenes")
print(f"  Available:        {n_scenes} scenes")

---
## Workflow 5: Automated Weekly Monitoring

In [ ]:
# Full automated monitoring workflow as Python function
# Run this weekly to track any AOI

def run_monitoring_workflow(
    name: str,
    bbox: str,
    output_dir: Path,
    providers: list = None,
    days_back: int = 14,
    cloud_max: float = 20.0,
    bands: list = None,
    max_scenes: int = 3,
):
    """
    Automated monitoring workflow:
    1. Search recent scenes in the AOI
    2. Download the best scenes
    3. Return a monitoring report
    """
    from datetime import date, timedelta

    providers = providers or ["aws_earth", "planetary_computer"]
    bands = bands or ["B04", "B08"]  # NDVI by default

    end_date = date.today()
    start_date = end_date - timedelta(days=days_back)

    print(f"\n{'='*60}")
    print(f"  Monitoring: {name}")
    print(f"  Period: {start_date} → {end_date}")
    print(f"  Bbox: {bbox}")
    print(f"{'='*60}")

    # Step 1: Search
    query = SearchQuery(
        bbox=BoundingBox.from_string(bbox),
        start_date=str(start_date),
        end_date=str(end_date),
        cloud_cover_max=cloud_max,
        max_results=max_scenes * 3,  # Get more, take the best
        sort_by="cloud_cover",
        sort_ascending=True,
    )

    results = client.search(query, providers=providers)
    print(f"\n[Search] Found {len(results)} scenes")

    if not results:
        return {"name": name, "status": "no_data", "scenes_found": 0}

    # Step 2: Select best scenes
    selected = results[:max_scenes]
    print(f"[Select] Using {len(selected)} best scenes:")
    for s in selected:
        dt = str(s.properties.get("datetime", ""))[:10] if s.properties else "?"
        print(f"  {dt}  cloud={s.cloud_cover:.0f}%  {s.id[:40]}")

    # Step 3: Download
    output_dir.mkdir(parents=True, exist_ok=True)
    options = DownloadOptions(
        parallel=2, resume=True, on_failure="skip",
        bands=bands,
    )

    dl_results = client.download(selected, output_dir, options)
    succeeded = [r for r in dl_results if r.success]
    total_mb = sum(r.bytes_downloaded or 0 for r in succeeded) / (1024*1024)

    print(f"[Download] {len(succeeded)}/{len(selected)} scenes downloaded ({total_mb:.1f} MB)")

    # Step 4: Report
    report = {
        "name": name,
        "status": "success" if succeeded else "failed",
        "period": f"{start_date} → {end_date}",
        "scenes_found": len(results),
        "scenes_downloaded": len(succeeded),
        "total_mb": round(total_mb, 1),
        "best_cloud_cover": round(selected[0].cloud_cover, 1) if selected else None,
        "providers_used": list(set(r.provider for r in results[:5])),
        "output_dir": str(output_dir),
    }

    print(f"[Report] Status: {report['status']}")
    return report


# Run monitoring for multiple AOIs
monitoring_sites = [
    ("NYC Area",       "-74.1,40.6,-73.7,40.9",  Path("output/monitoring/nyc")),
    ("Iowa Farmland",  "-93.8,41.9,-93.5,42.1",  Path("output/monitoring/iowa")),
    ("Houston Urban",  "-95.6,29.6,-95.2,30.0",  Path("output/monitoring/houston")),
]

reports = []
for name, bbox, outdir in monitoring_sites:
    report = run_monitoring_workflow(
        name=name, bbox=bbox, output_dir=outdir,
        days_back=30, cloud_max=20, max_scenes=2,
    )
    reports.append(report)

In [ ]:
# Monitoring report summary
print("\n" + "="*60)
print("  MONITORING REPORT SUMMARY")
print("="*60)
df_report = pd.DataFrame(reports)
print(df_report[[
    "name", "status", "scenes_found",
    "scenes_downloaded", "total_mb", "best_cloud_cover"
]].to_string(index=False))

# Save report
import json
report_path = Path("output/monitoring/report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, "w") as f:
    json.dump(reports, f, indent=2, default=str)
print(f"\n✓ Report saved → {report_path}")

---
## Summary

| Workflow | What we built | Key technique |
|---|---|---|
| NDVI Time Series | Monthly vegetation index over Iowa (2024) | Best-per-month selection + NDVI formula |
| Change Detection | Urban expansion 2020→2024 over Houston | Bi-temporal NDVI differencing |
| Multi-Sensor Fusion | Sentinel-2 + Landsat timeline over NYC | Multi-provider search + timeline plot |
| Cloud-Free Mosaic | Coverage probability analysis | Statistical P(cloud-free) model |
| Weekly Monitoring | 3-site automated pipeline | Reusable monitoring function |

**These are the building blocks for production earth observation systems.** Combine with PyGeoVision's GeoAI subsystems for automated inference on top of the data acquired here.